In [1]:
import sys
sys.path.append('../code')
from preprocessing import get_common_genes

/home/jwinchel/miniconda3/envs/heimdall/lib/python3.10/site-packages/anndata/utils.py:429: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/home/jwinchel/miniconda3/envs/heimdall/lib/python3.10/site-packages/anndata/utils.py:429: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
/home/jwinchel/miniconda3/envs/heimdall/lib/python3.10/site-packages/anndata/utils.py:429: FutureWarning: Importing read_hdf from `anndata` is deprecated. Import anndata.io.read_hdf instead.
  warnings.warn(msg, FutureWarning)
/home/jwinchel/miniconda3/envs/heimdall/lib/python3.10/site-packages/anndata/utils.py:429: FutureWarning: Importing read_loom from `anndata` is deprecated. Import anndata.io.read_loom instead.
  warnings.warn(msg, FutureWarning)
/home/jwinchel/miniconda3/envs/heimdall/lib/python3.10/site-packages/anndata/utils.py:

In [ ]:
import numpy as np
import os
import anndata as ad
import scanpy as sc
import pandas as pd
import pickle
import sys
sys.path.append('/home/jwinchel/Heimdall/')
from Heimdall.utils import convert_to_ensembl_ids

def load_nsclc_cosmx(data_dir,sc_file='/compute/oven-0-13/jwinchel/scRNA-seq/GSM3635278_human_p1t1_raw_counts.tsv',cell_meta_file='/compute/oven-0-13/jwinchel/NSCLC_ST_cell_metadata.csv',save_dir='/compute/oven-0-13/jwinchel/spatial_adata/'):
    # get spatial transcriptomics files
    st_files = sorted([os.path.join(data_dir,x,x+'-Flat_files_and_images/',x+"_exprMat_file.csv") for x in os.listdir(data_dir) if x.startswith('Lung')])
    met_files = sorted([os.path.join(data_dir,x,x+'-Flat_files_and_images/',x+"_metadata_file.csv") for x in os.listdir(data_dir) if x.startswith('Lung')])

    ad_list = []
    common_genes = get_common_genes(sc_file, st_files[0],data_dir)

    metas = []
    for i,f in enumerate(st_files):
        df = pd.read_csv(f)

        # read metadata
        met_df = pd.read_csv(met_files[i])
        met_df['tissue'] = "_".join(f.split('/')[-1].split('_')[:-2])
        metas.append(met_df)

        # separate metadata from gene counts
        obs = df[['fov', 'cell_ID']].copy()
        expr = df.drop(columns=['fov', 'cell_ID']).copy()

        # build AnnData
        st_sample = ad.AnnData(X=expr.values, obs=obs)
        st_sample.var_names = expr.columns
        st_sample.obs_names = obs['cell_ID'].astype(str)

        # convert genes to ensembl IDs
        st_sample, _ = convert_to_ensembl_ids(st_sample, data_dir)

        # restrict to common genes
        st_sample = st_sample[:, st_sample.var_names.isin(common_genes)].copy()
        st_sample.obs['tissue'] = f.split('/')[4]

        ad_list.append(st_sample)

    metadata = pd.concat(metas, join='outer')
    st_data = ad.concat(ad_list, join='outer')
    st_data.obs = st_data.obs.reset_index(drop=True)
    
    st_data.obs = st_data.obs.merge(metadata[['tissue','fov','cell_ID','CenterX_local_px','CenterY_local_px','CenterX_global_px','CenterY_global_px']],on=['tissue','fov','cell_ID'],how='left')

    st_data.obs['cell_ID'] = st_data.obs['cell_ID'].astype(str)

    cell_meta = pd.read_csv(cell_meta_file)
    cell_meta['cell_ID'] = cell_meta['cell_ID'].astype(str)
    cell_meta = cell_meta.rename(columns={'cell_ID':'cell_ID_long'})
    cell_meta['cell_ID'] = cell_meta['cell_ID_long'].str.split('_').str[-1].astype(str)
    st_data.obs = st_data.obs.merge(cell_meta.drop(columns=['tissue']),left_on=['tissue','fov','cell_ID'],right_on=['Run_Tissue_name','fov','cell_ID'],how='left')
    
    cat_cols = [c for c in st_data.obs.columns if st_data.obs[c].dtype in ['category', 'O']]
    st_data.obs[cat_cols] = st_data.obs[cat_cols].astype(str)

    st_data.obs.reset_index(drop=True, inplace=True)
    st_data.obs.index = st_data.obs.index.astype(str)
    st_data.obs["cell_type_merged"] = st_data.obs["cell_type"].map({"T CD4 naive": "T CD4+", 
                                                                    "T CD4 memory": "T CD4+", 
                                                                    "T CD8 naive": "T CD8+", 
                                                                    "T CD8 memory": "T CD8+",
                                                                    "pDC":"DC",
                                                                    "mDC":"DC"}).fillna(st_data.obs["cell_type"])


    st_data.obs = st_data.obs.rename(columns={'cell_type_merged':'cell_type','cell_type':'cell_type_orig'})
    st_data.obs['cell_type'] = st_data.obs['cell_type'].apply(lambda x: 'tumor' if x.startswith('tumor') else x)
    
    tissue_list = st_data.obs['tissue'].unique()
    for tissue in tissue_list:
        tissue_adata = st_data[st_data.obs['tissue'] == tissue]
        # tissue_adata.obs = tissue_adata.obs.fillna('NA')
        tissue_adata.write_h5ad(os.path.join(save_dir, f'cosmx_nsclc_{tissue.replace("_","-")}_adata.h5ad'))
        tissue_adata = tissue_adata[tissue_adata.obs['cell_type']!='NA']
        tissue_adata.write_h5ad(os.path.join(save_dir, f'cosmx_nsclc_{tissue.replace("_","-")}_adata_annotated.h5ad'))

    return st_data

In [3]:
adata = load_nsclc_cosmx('/compute/oven-0-13/jwinchel/')

/home/jwinchel/miniconda3/envs/heimdall/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


human
/compute/oven-0-13/jwinchel
Mapping data directory: /compute/oven-0-13/jwinchel/gene_mapping/ensembl/human
Loading mapping from cache: /compute/oven-0-13/jwinchel/gene_mapping/ensembl/human/symbol_to_ensembl.json
Successfully mapped 957 out of 980 identifiers (97.7%)


/home/jwinchel/miniconda3/envs/heimdall/lib/python3.10/site-packages/anndata/_core/anndata.py:1757: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/jwinchel/miniconda3/envs/heimdall/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


human
/compute/oven-0-13/jwinchel
Mapping data directory: /compute/oven-0-13/jwinchel/gene_mapping/ensembl/human
Loading mapping from cache: /compute/oven-0-13/jwinchel/gene_mapping/ensembl/human/symbol_to_ensembl.json
Successfully mapped 957 out of 980 identifiers (97.7%)


/home/jwinchel/miniconda3/envs/heimdall/lib/python3.10/site-packages/anndata/_core/anndata.py:1757: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/jwinchel/miniconda3/envs/heimdall/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


human
/compute/oven-0-13/jwinchel
Mapping data directory: /compute/oven-0-13/jwinchel/gene_mapping/ensembl/human
Loading mapping from cache: /compute/oven-0-13/jwinchel/gene_mapping/ensembl/human/symbol_to_ensembl.json
Successfully mapped 957 out of 980 identifiers (97.7%)


... (7 outputs left out)

human
/compute/oven-0-13/jwinchel
Mapping data directory: /compute/oven-0-13/jwinchel/gene_mapping/ensembl/human
Loading mapping from cache: /compute/oven-0-13/jwinchel/gene_mapping/ensembl/human/symbol_to_ensembl.json
Successfully mapped 957 out of 980 identifiers (97.7%)


/home/jwinchel/miniconda3/envs/heimdall/lib/python3.10/site-packages/anndata/_core/anndata.py:1757: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/jwinchel/miniconda3/envs/heimdall/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


human
/compute/oven-0-13/jwinchel
Mapping data directory: /compute/oven-0-13/jwinchel/gene_mapping/ensembl/human
Loading mapping from cache: /compute/oven-0-13/jwinchel/gene_mapping/ensembl/human/symbol_to_ensembl.json
Successfully mapped 957 out of 980 identifiers (97.7%)


/home/jwinchel/miniconda3/envs/heimdall/lib/python3.10/site-packages/anndata/_core/anndata.py:1757: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/jwinchel/miniconda3/envs/heimdall/lib/python3.10/site-packages/anndata/_core/anndata.py:1757: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/jwinchel/miniconda3/envs/heimdall/lib/python3.10/site-packages/anndata/_core/anndata.py:1145: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  df[key] = c
/home/jwinchel/miniconda3/envs/heimdall/lib/python3.10/site-packages/anndata/_core/anndata.py:1145: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  df[key] = c
/home/jwinchel/miniconda3/envs/heimdall/lib/python3.10/site-packages/anndata/_core/anndata.py:1145

In [4]:
adata.obs

,fov,cell_ID,tissue,CenterX_local_px,CenterY_local_px,CenterX_global_px,CenterY_global_px,sdimx,sdimy,cell_ID_long,...,Diversity,totalcounts,log10totalcounts,background,remove_flagged_cells,patient,cell_type_orig,niche,prop_tumor_in_100_neighbors,cell_type
0,1,0,Lung12,NaN,NaN,NaN,NaN,NaN,NaN,nan,...,NaN,NaN,NaN,NaN,nan,nan,nan,nan,NaN,nan
1,1,1,Lung12,553.0,3625.0,-4424.777778,-156608.333333,0.09954,-3.94614,c_7_1_1,...,80.0,108.0,2.033424,0.023273,False,Lung12,neutrophil,tumor interior,0.77,neutrophil
2,1,2,Lung12,819.0,3626.0,-4158.777778,-156607.333333,0.14742,-3.94596,c_7_1_2,...,49.0,70.0,1.845098,0.015084,False,Lung12,tumor 12,tumor interior,0.74,tumor
3,1,3,Lung12,1102.0,3616.0,-3875.777778,-156617.333333,0.19836,-3.94776,c_7_1_3,...,74.0,93.0,1.968483,0.020040,False,Lung12,tumor 12,tumor interior,0.72,tumor
4,1,4,Lung12,1392.0,3596.0,-3585.777778,-156637.333333,0.25056,-3.95136,c_7_1_4,...,209.0,395.0,2.596597,0.085117,False,Lung12,tumor 12,tumor interior,0.62,tumor
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
800554,45,3040,Lung9_Rep2,1336.0,17.0,4274.888889,339.222222,NaN,NaN,nan,...,NaN,NaN,NaN,NaN,nan,nan,nan,nan,NaN,nan
800555,45,3041,Lung9_Rep2,1387.0,13.0,4325.888889,335.222222,NaN,NaN,nan,...,NaN,NaN,NaN,NaN,nan,nan,nan,nan,NaN,nan
800556,45,3042,Lung9_Rep2,268.0,13.0,3206.888889,335.222222,NaN,NaN,nan,...,NaN,NaN,NaN,NaN,nan,nan,nan,nan,NaN,nan
800557,45,3043,Lung9_Rep2,3542.0,9.0,6480.888889,331.222222,NaN,NaN,nan,...,NaN,NaN,NaN,NaN,nan,nan,nan,nan,NaN,nan


In [4]:
data_dir = '/compute/oven-0-13/jwinchel/'
st_files = sorted([os.path.join(data_dir,x,x+'-Flat_files_and_images/',x+"_metadata_file.csv") for x in os.listdir(data_dir) if x.startswith('Lung')])
mets = []
for f in st_files:
    met_df = pd.read_csv(f)
    met_df['tissue'] = "_".join(f.split('/')[-1].split('_')[:-2])
    mets.append(met_df)
metadata = pd.concat(mets, join='outer')
test_adata = adata.copy()
test_adata.obs = test_adata.obs.merge(metadata[['tissue','fov','cell_ID','CenterX_local_px','CenterY_local_px','CenterX_global_px','CenterY_global_px']],on=['tissue','fov','cell_ID'],how='left')


/home/jwinchel/miniconda3/envs/heimdall/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


In [5]:
test_adata.obs

,fov,cell_ID,tissue,CenterX_local_px,CenterY_local_px,CenterX_global_px,CenterY_global_px
0,1,0,Lung12,NaN,NaN,NaN,NaN
1,1,1,Lung12,553.0,3625.0,-4424.777778,-156608.333333
2,1,2,Lung12,819.0,3626.0,-4158.777778,-156607.333333
3,1,3,Lung12,1102.0,3616.0,-3875.777778,-156617.333333
4,1,4,Lung12,1392.0,3596.0,-3585.777778,-156637.333333
...,...,...,...,...,...,...,...
800554,45,3040,Lung9_Rep2,1336.0,17.0,4274.888889,339.222222
800555,45,3041,Lung9_Rep2,1387.0,13.0,4325.888889,335.222222
800556,45,3042,Lung9_Rep2,268.0,13.0,3206.888889,335.222222
800557,45,3043,Lung9_Rep2,3542.0,9.0,6480.888889,331.222222


In [6]:
cell_meta = pd.read_csv('/compute/oven-0-13/jwinchel/NSCLC_ST_cell_metadata.csv')

In [ ]:
# merge_keys = list(set(cell_meta.columns).intersection(set(test_adata.obs.columns)))
# merge_keys
test_adata.obs['cell_ID'] = test_adata.obs['cell_ID'].astype(str)
cell_meta['cell_ID'] = cell_meta['cell_ID'].astype(str)
cell_meta = cell_meta.rename(columns={'cell_ID':'cell_ID_long'})
cell_meta['cell_ID'] = cell_meta['cell_ID_long'].str.split('_').str[-1].astype(str)
test_adata.obs = test_adata.obs.merge(cell_meta.drop(columns=['tissue']),left_on=['tissue','fov','cell_ID'],right_on=['Run_Tissue_name','fov','cell_ID'],how='left')

In [8]:
test_adata.obs['cell_ID_long'].value_counts()
cat_cols = [c for c in test_adata.obs.columns if test_adata.obs[c].dtype in ['category', 'O']]
print(cat_cols)
test_adata.obs[cat_cols] = test_adata.obs[cat_cols].astype(str)

['cell_ID', 'tissue', 'cell_ID_long', 'dualfiles', 'Slide_name', 'Run_name', 'ISH.concentration', 'Dash', 'Run_Tissue_name', 'Panel', 'remove_flagged_cells', 'patient', 'cell_type', 'niche']


In [9]:
test_adata.obs.reset_index(drop=True, inplace=True)
test_adata.obs.index = test_adata.obs.index.astype(str)
test_adata.obs["cell_type_merged"] = test_adata.obs["cell_type"].map({"T CD4 naive": "T CD4+", 
                                                                "T CD4 memory": "T CD4+", 
                                                                "T CD8 naive": "T CD8+", 
                                                                "T CD8 memory": "T CD8+",
                                                                "pDC":"DC",
                                                                "mDC":"DC"}).fillna(test_adata.obs["cell_type"])


test_adata.obs = test_adata.obs.rename(columns={'cell_type_merged':'cell_type','cell_type':'cell_type_orig'})
test_adata.obs['cell_type'] = test_adata.obs['cell_type'].apply(lambda x: 'tumor' if x.startswith('tumor') else x)

In [16]:
tissue_list = sorted(test_adata.obs['tissue'].unique())
save_dir = '/compute/oven-0-13/jwinchel/spatial_adata/'
for tissue in tissue_list:
    tissue_adata = test_adata[test_adata.obs['tissue'] == tissue]
    # tissue_adata.obs = tissue_adata.obs.fillna('NA')
    tissue_adata.write_h5ad(os.path.join(save_dir, f'cosmx_nsclc_{tissue.replace("_","-")}_adata.h5ad'))
    tissue_adata = tissue_adata[tissue_adata.obs['cell_type']!='NA']
    tissue_adata.write_h5ad(os.path.join(save_dir, f'cosmx_nsclc_{tissue.replace("_","-")}_adata_annotated.h5ad'))

/home/jwinchel/miniconda3/envs/heimdall/lib/python3.10/site-packages/anndata/_core/anndata.py:1145: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  df[key] = c
/home/jwinchel/miniconda3/envs/heimdall/lib/python3.10/site-packages/anndata/_core/anndata.py:1145: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  df[key] = c
/home/jwinchel/miniconda3/envs/heimdall/lib/python3.10/site-packages/anndata/_core/anndata.py:1145: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  df[key] = c
/home/jwinchel/miniconda3/envs/heimdall/lib/python3.10/site-packages/anndata/_core/anndata.py:1145: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  df[key] = c
/home/jwinchel/miniconda3/envs/heimdall/lib/python3.10/site-packages/anndata/_core/anndata.py:1145: ImplicitModificationWarning: Trying to m

In [ ]:
# # genes never expressed
# obj_cols = [c for c in adata.obs.columns if adata.obs[c].dtype== 'object']
# adata.obs[obj_cols] = adata.obs[obj_cols].fillna('NA').astype(str)
# adata.write_h5ad('/compute/oven-0-13/jwinchel/scRNA-seq/adata/GSE127465_tumor.h5ad')

In [ ]:
# meta['Barcode_Library'] = meta['Barcode'] + '_' + meta['Library']

# adata.obs.merge(meta, left_on='cell_ID_sample', right_on='Barcode_Library').columns